#### Loading the packages and pre-build models and making the input data ready

In [1]:
## Importing the necessary packages
import pandas as pd
import numpy as np
import joblib
import shap

In [2]:
## Loading all the models
logistic_regression_model = joblib.load("../../../models/heart/old_models/old_logistic_regression_model.pkl")
random_forest_model = joblib.load("../../../models/heart/old_models/old_random_forest_model.pkl")
encoder = joblib.load("../../../models/heart/old_models/old_one_hot_encoder.pkl")
scaler = joblib.load("../../../models/heart/old_models/old_standard_scaler.pkl")

In [3]:
## Getting the input frame ready
input_data = pd.DataFrame([{

    "Age": 75,
    "Gender": "Female",
    "Cholesterol": 228,
    "Blood Pressure": 119,
    "Heart Rate": 66,
    "Smoking": "Current",
    "Alcohol Intake": "Heavy",
    "Exercise Hours": 1,
    "Family History": "No",
    "Diabetes": "No",
    "Obesity": "Yes",
    "Stress Level": 8,
    "Blood Sugar": 119,
    "Exercise Induced Angina": "Yes",
    "Chest Pain Type": "Atypical Angina"

}])
original_input_data = input_data.copy()
gender_map = {'Female':0,'Male':1} 
binary_map = {'Yes':1,'No':0}
bin_cols = ['Family History','Diabetes','Obesity','Exercise Induced Angina']
one_hot_cols = ['Smoking','Alcohol Intake','Chest Pain Type']
num_cols = ['Age','Cholesterol','Blood Pressure','Heart Rate','Exercise Hours','Stress Level','Blood Sugar']
input_data['Gender'] = input_data['Gender'].map(gender_map)
for col in bin_cols:
    input_data[col] = input_data[col].map(binary_map)
encoded_array = encoder.transform(input_data[one_hot_cols])
encoded_df = pd.DataFrame(encoded_array,columns=encoder.get_feature_names_out(one_hot_cols))
input_data = input_data.drop(columns = one_hot_cols)
input_data = pd.concat([input_data,encoded_df],axis=1)
input_data[num_cols] = scaler.transform(input_data[num_cols])
input_data.head()

,Age,Gender,Cholesterol,Blood Pressure,Heart Rate,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Smoking_Former,Smoking_Never,Alcohol Intake_Moderate,Alcohol Intake_Unknown,Chest Pain Type_Atypical Angina,Chest Pain Type_Non-anginal Pain,Chest Pain Type_Typical Angina
0,1.460969,0,-0.396643,-0.622368,-1.166438,-1.194375,0,0,1,0.79331,-0.45173,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0


#### For the explainability of the logistic regression model

In [4]:
lr_prediction = logistic_regression_model.predict(input_data)[0]
lr_probabilty = logistic_regression_model.predict_proba(input_data)[0][1]
# =====================================================
# CHECK MODEL COEFFICIENTS
# =====================================================

feature_names = input_data.columns

coefficients = logistic_regression_model.coef_[0]

coef_df = pd.DataFrame({

    "Feature": feature_names,

    "Coefficient": coefficients

})

print(

    coef_df.sort_values(

        by="Coefficient",

        ascending=False

    )

)


lr_result = "YES" if lr_prediction==1 else "NO"
print(f"Heart Disease : {lr_result}\n")
print(f"Risk Probability : {lr_probabilty:.2f}\n")
if lr_probabilty>=0.70:
    print(f"Recommendation : Immediate doctor consultation is advised.")
elif lr_probabilty>=0.40:
    print(f"Recommendation : Regular doctor consultation is advised.")
else:
    print(f"Recommendation : No immediate consultation needed.")

## Building the explainer model for the logistic regression
X_train = pd.read_csv("../../../data/heart/processed/old_dataset_synthetic/X_train.csv")
lr_explainer = shap.Explainer(logistic_regression_model,X_train)

## For the particular input - output the explanation
lr_shap_values = lr_explainer(input_data)
print("\nExplanation : \n")
for i,col in enumerate(input_data.columns):
    impact = lr_shap_values.values[0][i]
    if col in original_input_data.columns:
        value = original_input_data.iloc[0][col]
        if impact>0:
            print(f"{col} : {value} , Increased risk of heart disease.")
        elif impact<0:
            print(f"{col} : {value} , Decreased risk of heart disease.")
    else:
        split_col = col.split("_")
        original_feature = split_col[0]
        encoded_value = "_".join(split_col[1:])
        if input_data.iloc[0][col] == 1:
            if impact > 0:
                print(f"{original_feature} : {encoded_value} , Increased risk of heart disease.")
            elif impact < 0:
                print(
                    f"{original_feature} : {encoded_value} , Decreased risk of heart disease.")

                             Feature  Coefficient
0                                Age     3.107040
2                        Cholesterol     1.950186
6                     Family History     0.398163
15            Alcohol Intake_Unknown     0.235331
18    Chest Pain Type_Typical Angina     0.176322
14           Alcohol Intake_Moderate     0.123947
1                             Gender     0.064010
13                     Smoking_Never     0.052605
9                       Stress Level     0.050900
10                       Blood Sugar     0.019548
16   Chest Pain Type_Atypical Angina     0.006700
4                         Heart Rate    -0.000448
7                           Diabetes    -0.019588
5                     Exercise Hours    -0.098497
3                     Blood Pressure    -0.100115
8                            Obesity    -0.177330
11           Exercise Induced Angina    -0.228908
17  Chest Pain Type_Non-anginal Pain    -0.238432
12                    Smoking_Former    -0.283502


#### For the explainability of the random forest model

In [5]:
rf_prediction = random_forest_model.predict(input_data)[0]
rf_probabilty = random_forest_model.predict_proba(input_data)[0][1]
rf_result = "YES" if rf_prediction==1 else "NO"
print(f"Heart Disease : {rf_result}\n")
print(f"Risk Probability : {rf_probabilty:.2f}\n")
if rf_probabilty>=0.70:
    print(f"Recommendation : Immediate doctor consultation is advised.")
elif rf_probabilty>=0.40:
    print(f"Recommendation : Regular doctor consultation is advised.")
else:
    print(f"Recommendation : No immediate consultation needed.")

## Building the explainer model for the logistic regression
rf_explainer = shap.TreeExplainer(random_forest_model)

## For the particular input - output the explanation
rf_shap_values = rf_explainer.shap_values(input_data)
rf_values = rf_shap_values[:,:,1]
print("\nExplanation : \n")
for i,col in enumerate(input_data.columns):
    impact = rf_values[0][i]
    if col in original_input_data.columns:
        value = original_input_data.iloc[0][col]
        if impact>0:
            print(f"{col} : {value} , Increased risk of heart disease.")
        elif impact<0:
            print(f"{col} : {value} , Decreased risk of heart disease.")
    else:
        split_col = col.split("_")
        original_feature = split_col[0]
        encoded_value = "_".join(split_col[1:])
        if input_data.iloc[0][col] == 1:
            if impact > 0:
                print(f"{original_feature} : {encoded_value} , Increased risk of heart disease.")
            elif impact < 0:
                print(
                    f"{original_feature} : {encoded_value} , Decreased risk of heart disease.")

Heart Disease : YES

Risk Probability : 0.96

Recommendation : Immediate doctor consultation is advised.

Explanation : 

Age : 75 , Increased risk of heart disease.
Gender : Female , Increased risk of heart disease.
Cholesterol : 228 , Increased risk of heart disease.
Blood Pressure : 119 , Increased risk of heart disease.
Heart Rate : 66 , Increased risk of heart disease.
Exercise Hours : 1 , Increased risk of heart disease.
Family History : No , Decreased risk of heart disease.
Diabetes : No , Increased risk of heart disease.
Obesity : Yes , Decreased risk of heart disease.
Stress Level : 8 , Increased risk of heart disease.
Blood Sugar : 119 , Increased risk of heart disease.
Exercise Induced Angina : Yes , Decreased risk of heart disease.
Chest Pain Type : Atypical Angina , Decreased risk of heart disease.


#### Saving the explainer models

In [6]:
joblib.dump(lr_explainer,"../../../models/heart/old_models/old_lr_shap_explainer.pkl")
joblib.dump(rf_explainer,"../../../models/heart/old_models/old_rf_shap_explainer.pkl")

['../../../models/heart/old_models/old_rf_shap_explainer.pkl']

In [7]:
print(input_data.columns.tolist())
print(X_train.columns.tolist())

['Age', 'Gender', 'Cholesterol', 'Blood Pressure', 'Heart Rate', 'Exercise Hours', 'Family History', 'Diabetes', 'Obesity', 'Stress Level', 'Blood Sugar', 'Exercise Induced Angina', 'Smoking_Former', 'Smoking_Never', 'Alcohol Intake_Moderate', 'Alcohol Intake_Unknown', 'Chest Pain Type_Atypical Angina', 'Chest Pain Type_Non-anginal Pain', 'Chest Pain Type_Typical Angina']
['Age', 'Gender', 'Cholesterol', 'Blood Pressure', 'Heart Rate', 'Exercise Hours', 'Family History', 'Diabetes', 'Obesity', 'Stress Level', 'Blood Sugar', 'Exercise Induced Angina', 'Smoking_Former', 'Smoking_Never', 'Alcohol Intake_Moderate', 'Alcohol Intake_Unknown', 'Chest Pain Type_Atypical Angina', 'Chest Pain Type_Non-anginal Pain', 'Chest Pain Type_Typical Angina']
